# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is published in [FAIR^2 - SenScience](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and its schema is available at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

All dataset entities (record sets, fields, columns) are referenced by their `@id` fields to ensure reproducibility and disambiguation.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load dataset and its metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets and fields. All entities are referenced by their `@id`.

We'll inspect the available record sets and their structure.

In [ ]:
# List record sets with their @id and fields
record_sets = list(metadata.record_sets)

if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rec in record_sets:
        print(f"Record set @id: {rec.id}")
        print(f"  Name: {getattr(rec, 'name', '')}")
        print(f"  Description: {getattr(rec, 'description', '')}")
        print("  Fields and columns:")
        for field in rec.fields:
            print(f"    Field @id: {field.id} (name: {getattr(field, 'name', '')})")
            if hasattr(field, 'columns'):
                for col in field.columns:
                    print(f"      Column @id: {col.id} (name: {getattr(col, 'name', '')})")
        print("\n")

# For demonstration, select the first record set if present
selected_record_set_id = record_sets[0].id if record_sets else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Change the variable `selected_record_set_id` to select a different record set by its `@id`.

In [ ]:
# Load records from the selected record set
if selected_record_set_id:
    print(f"Loading records from record set @id: {selected_record_set_id}")
    records = list(dataset.records(record_set=selected_record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records.")
    print("Columns:", df.columns.tolist())
    display(df.head())
else:
    print("No record set available to load data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. The fields are accessed by their column names (`@id` is used for mapping, but column names are used in DataFrames).

We'll select a numeric field (e.g., coverage percentage, molecular weight, or peptide count) for analysis.

Adjust `numeric_field_id` and `group_field_id` to the desired numeric and grouping field `@id` from the record set listing above.

In [ ]:
# Choose the column names you wish to analyze. Replace these with @id/column names from the overview above.
numeric_field_candidates = [col for col in df.columns if df[col].dtype != object]
print('Numeric candidates detected in DataFrame:', numeric_field_candidates)

# Example: Use 'coverage' or a molecular weight field if present.
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else None
group_field_candidates = [col for col in df.columns if df[col].dtype == object]
group_field = group_field_candidates[0] if group_field_candidates else None

if numeric_field:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())
    
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Now group by a categorical field
    if group_field:
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No string/categorical fields found for grouping.")
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(12, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
In this notebook, you loaded a dataset from a Croissant schema using `mlcroissant`, inspected its structure using `@id` references, performed basic exploratory data analysis on numeric and categorical fields, and visualized key distributions. This workflow can be adapted to other Croissant datasets by referring to their record set and field `@id`s.

*Remember: always check the column names and @id values in the overview when adapting processing code for different datasets or record sets!*